# CAMEL Ambient Solar Wind Validation

## Overview

This notebook demonstrates how to validate ambient solar wind models against in situ measurements near Earth. The workflow is based on the **CAMEL (Comprehensive Assessment of Models and Events using Library tools)** framework developed at the [Community Coordinated Modeling Center (CCMC)](https://ccmc.gsfc.nasa.gov/).

This effort is part of the international [ISWAT H1-01 Team](https://iswat-cospar.org/h1-01), which aims to establish a unified validation approach for solar wind models across the heliophysics and space weather communities.

The validation procedures presented in this notebook series are also available through an interactive web application built with Plotly Dash and Python. The **Ambient Solar Wind Validation Dashboard** can be accessed here:

- [CCMC CAMEL Ambient Solar Wind Validation Dashboard](https://ccmc.gsfc.nasa.gov/camel/AmbientSolarWind/home)

These notebooks provide a reproducible version of the same validation workflow and are intended to help users explore, reproduce, and extend the validation procedures of the CCMC CAMEL dashboard.

**Notebook version:** 1.0

---

## Notebook Scope

### 05 Multi-Model Comparison

This notebook:
- loads the preprocessed example datasets generated in `01_api_and_preprocessing.ipynb`
- prepares multiple model outputs for side-by-side comparison against the same observational reference
- reproduces the multi-model visualization workflow used in the dashboard

Multi-model comparison visualizations help evaluate model performance side by side and provide context for comparing model behavior against a simple persistence baseline.

---

## Expected Inputs

This notebook expects the following processed datasets:

```text
../example_data/processed/
├── omni_obs.csv
├── ambsowi_mod.csv
├── bravda_mod.csv
└── p27_mod.csv
```

## Imports

We import the metric and visual diagnostic functions from `utils/`. Note that these functions are an exact copy of the functions that are used in the CAMEL Ambient Solar Wind Dashboard. 

In [ ]:
import sys
from pathlib import Path

project_root = ".."
utils_path = "../utils"

if project_root not in sys.path:
    sys.path.append(project_root)
if utils_path not in sys.path:
    sys.path.append(utils_path)

import numpy as np
import pandas as pd
from IPython.display import display

from utils.skill_scores import *
from utils.visualizations import *

## Load the processed example datasets

These CSV files are assumed to have already been preprocessed and aligned in `01_api_and_preprocessing.ipynb`. Note that if you want to change the selected time window for validation you need to go back to `01_api_and_preprocessing.ipynb`.

In [ ]:
data_dir = Path("../example_data/processed")

omni_obs = pd.read_csv(data_dir / "omni_obs.csv", parse_dates=["Time"], index_col="Time")
ambsowi_mod = pd.read_csv(data_dir / "ambsowi_mod.csv", parse_dates=["Time"], index_col="Time")
bravda_mod = pd.read_csv(data_dir / "bravda_mod.csv", parse_dates=["Time"], index_col="Time")
p27_mod = pd.read_csv(data_dir / "p27_mod.csv", parse_dates=["Time"], index_col="Time")

print("Observed shape:   ", omni_obs.shape)
print("AmbSoWi-ML shape: ", ambsowi_mod.shape)
print("BRaVDA shape:     ", bravda_mod.shape)
print("Persistence shape:", p27_mod.shape)


## Prepare inputs for multi-model comparison

The multi-model visualization helpers expect a list of standardized long-format model DataFrames and one corresponding observational DataFrame.


In [ ]:
def to_pred(df: pd.DataFrame, name: str) -> pd.DataFrame:
    d = df.copy().reset_index()
    d = d.rename(columns={"Flow_Speed": "Model"})
    d["Name"] = name
    return d[["Time", "Model", "Name"]]

def to_obs(df: pd.DataFrame, name: str) -> pd.DataFrame:
    d = df.copy().reset_index()
    d = d.rename(columns={"Flow_Speed": "Obs"})
    d["Name"] = name
    return d[["Time", "Obs", "Name"]]

observed_data = to_obs(omni_obs, "OMNI2 | Hourly averaged")

predicted_ambsowi = to_pred(ambsowi_mod, "AmbSoWi-ML")
predicted_bravda = to_pred(bravda_mod, "BRaVDA")
baseline_data = to_pred(p27_mod, "Persistence (27-days)")

predicted_data_list = [
    predicted_ambsowi,
    predicted_bravda,
    baseline_data,
]

## Summary Metrics Table

This table provides a compact quantitative overview before generating the multi-model visualizations.

In [ ]:
def compute_metric_table(models: dict, observed: pd.DataFrame) -> pd.DataFrame:
    rows = {}
    for name, model_df in models.items():
        rows[name] = {
            "ME": float(np.squeeze(meScore(model_df, observed))),
            "MAE": float(np.squeeze(maeScore(model_df, observed))),
            "MSE": float(np.squeeze(meanSquaredError(model_df, observed))),
            "RMSE": float(np.squeeze(rmsScore(model_df, observed))),
            "CC": float(np.squeeze(ccScore(model_df, observed))),
        }
    return pd.DataFrame.from_dict(rows, orient="index")

metrics_table = compute_metric_table(
    {
        "AmbSoWi-ML": ambsowi_mod,
        "BRaVDA": bravda_mod,
        "Persistence (27d)": p27_mod,
    },
    omni_obs,
)

display(metrics_table.round(2))

## Visual Diagnostics for Multi-Model Comparison

Multi-model comparison visualizations allow several model outputs to be evaluated side by side against the same observational dataset.

The examples in this section include:
- a multi-model time series comparison
- a multi-model distribution comparison
- a multi-model RMSE timeline


In [ ]:
fig = create_mm_line_chart(predicted_data_list, observed_data)
fig.update_layout(width=1200, height=500)
fig.show()


In [ ]:
fig = create_mm_box_plot(predicted_data_list, observed_data)
fig.update_layout(width=900, height=520)
fig.show()

In [ ]:
fig = create_mm_rmse_timeline(
    predicted_data_list,
    observed_data,
    baseline_data,
    normalize_time="naive_utc",
    floor="H",
)
fig.update_layout(width=1200, height=500)
fig.show()

## Export multi-model figures

Save all multi-model visualizations generated in this notebook as PDF files in `../results/output_pdfs/`. This requires `kaleido` for static Plotly image export.


In [ ]:
out_dir = Path("../results/output_pdfs")
out_dir.mkdir(parents=True, exist_ok=True)

figures = {
    "multi_model_line_chart": create_mm_line_chart(
        predicted_data_list,
        observed_data,
    ),
    "multi_model_box_plot": create_mm_box_plot(
        predicted_data_list,
        observed_data,
    ),
    "multi_model_rmse_timeline": create_mm_rmse_timeline(
        predicted_data_list,
        observed_data,
        baseline_data,
        normalize_time="naive_utc",
        floor="H",
    ),
}

figure_sizes = {
    "multi_model_line_chart": (1200, 500),
    "multi_model_box_plot": (900, 520),
    "multi_model_rmse_timeline": (1200, 500),
}

for name, fig in figures.items():
    width, height = figure_sizes[name]
    fig.update_layout(width=width, height=height)

    pdf_path = out_dir / f"{name}.pdf"

    # Requires kaleido:
    fig.write_image(pdf_path)

    print("Saved:", pdf_path)